In [ ]:
import pandas as pd
import xarray as xr

dt: xr.DataTree = xr.open_datatree("sample-data/results.zarr", engine="zarr")

overview: xr.DataTree = dt["overview"]  # ty: ignore[invalid-assignment]

combined: list[pd.DataFrame] = []
for i in overview.descendants:
    for k in i.descendants:
        ds = k.to_dataset()
        combined.append(ds.to_dataframe())


tmp = pd.concat(combined)


test_nodes: pd.Series = tmp["on_node"]


def change_type(x: str) -> int:
    tmp = x.split("l")
    return int(tmp[1])


res = test_nodes.map(change_type)
res = res.rename("node_num")
tmp["node_num"] = res


In [ ]:
from itertools import product
from typing import Any

tunable_params: pd.DataFrame = tmp[
    [
        "access_kind",
        "engine",
        "format",
        "filesize_per_chunk",
        "language",
        "num_nodes",
        "num_ranks",
        "no_caching",
        "parallel",
        "parallel_backend",
        "task_type",
        "total_filesize",
        "unit_var",
        "unit_chunk",
    ]
]

gathered: dict[str, list[Any]] = {}
for name, feature in tunable_params.items():
    gathered[str(name)] = list(feature.unique())

gathered["access_kind"].append("collective")
gathered["language"].append("c")
gathered["total_filesize"].extend([1, 5, 20, 30, 40, 60, 70, 80, 90, 100])
gathered["unit_var"].append("TB")
gathered["no_caching"].append(False)
gathered["task_type"].append("write")

comb = list(product(*gathered.values()))
comb_df = pd.DataFrame(
    comb,
    columns=[
        "access_kind",
        "engine",
        "format",
        "filesize_per_chunk",
        "language",
        "num_nodes",
        "num_ranks",
        "no_caching",
        "parallel",
        "parallel_backend",
        "task_type",
        "total_filesize",
        "unit_var",
        "unit_chunk",
    ],
)

parallel_impossible_backend_combo = comb_df[
    (comb_df["parallel"] == False) & (comb_df["parallel_backend"].notnull())
]
comb_df.drop(parallel_impossible_backend_combo.index, inplace=True)

parallel_impossible_backend_combo = comb_df[
    (comb_df["parallel"] == True) & (comb_df["parallel_backend"].isnull())
]
comb_df.drop(parallel_impossible_backend_combo.index, inplace=True)
print(len(comb_df))

In [ ]:
from collections import Counter

import plotly.express as px

fig = px.scatter(
    data_frame=tmp.sort_values(by=["on_node"]),
    x="engine",
    y="run_time",
    # size    = "occured",
    color="on_node",
    # error_y = "error bar",
)
fig.update_layout(scattermode="group")
fig.show()


counter: Counter[str] = Counter(tmp["on_node"])
df_count = pd.DataFrame(data={"nodes": counter.keys(), "occurred": counter.values()})


fig = px.bar(
    data_frame=df_count,
    y="occurred",
    x="nodes",
    color="nodes",
)
fig.show()

fig = px.line(
    data_frame=tmp,
    log_y=True,
    x="total_filesize",
    y="mean_time",
    color="engine",
    markers=True,
    hover_data=["format"],
    labels={"mean_time": "mean time in seconds (s)"},
    error_y="time_error_bar",
    color_discrete_map={
        "zarr-py": "#56B4E9",
        "netcdf4-py": "#4B0092",
        "hdf5-py": "#117733",
        "netcdf4-py-parallel": "#FFB000",
        "hdf5-py-parallel": "#FF4430",
    },
)
fig.show()

unit: str = tmp["unit_var"].iat[0]
fig = px.bar(
    data_frame=tmp,
    x="total_filesize",
    y="mean_throughput",
    color="engine",
    barmode="group",
    hover_data=["format"],
    labels={"mean_throughput": f"throughput in {unit}"},
    color_discrete_map={
        "zarr-py": "#56B4E9",
        "netcdf4-py": "#4B0092",
        "hdf5-py": "#117733",
        "netcdf4-py-parallel": "#FFB000",
        "hdf5-py-parallel": "#FF4430",
    },
)

fig.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

common_params = {
    "max_iter": 1_000,
    "learning_rate": 0.3,
    "validation_fraction": 0.2,
    "random_state": 42,
    "categorical_features": None,
    "scoring": "neg_root_mean_squared_error",
}

In [ ]:
hgbt_format = HistGradientBoostingRegressor(early_stopping=True, **common_params)

le = preprocessing.LabelEncoder()

X: pd.DataFrame = tmp.drop(columns=["format", "engine"])
y: pd.Series = le.fit_transform(tmp["format"])
X_format_train, X_format_test, y_format_train, y_format_test = train_test_split(
    X, y, train_size=0.33, random_state=42
)


hgbt_format.fit(
    X_format_train.apply(le.fit_transform), le.fit_transform(y_format_train)
)

_, ax1 = plt.subplots()
plt.plot(-hgbt_format.validation_score_)
_ = ax1.set(
    xlabel="number of iterations",
    ylabel="root mean squared error",
    title=f"Loss of hgbt with early stopping (n_iter={hgbt_format.n_iter_})",
)

test_accuracy = hgbt_format.score(
    X_format_test.apply(le.fit_transform), le.fit_transform(y_format_test)
)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
hgbt_throughput = HistGradientBoostingRegressor(early_stopping=True, **common_params)

le2 = preprocessing.LabelEncoder()

X: pd.DataFrame = tmp.drop(columns=["mean_throughput"])
y: pd.Series = le.fit_transform(tmp["mean_throughput"])
X_throughput_train, X_throughput_test, y_throughput_train, y_throughput_test = (
    train_test_split(X, y, train_size=0.33, random_state=42)
)

hgbt_throughput.fit(
    X_throughput_train.apply(le2.fit_transform), le2.fit_transform(y_throughput_train)
)

_, ax2 = plt.subplots()
plt.plot(-hgbt_throughput.validation_score_)
_ = ax2.set(
    xlabel="number of iterations",
    ylabel="root mean squared error",
    title=f"Loss of hgbt with early stopping (n_iter={hgbt_throughput.n_iter_})",
)

test_accuracy = hgbt_throughput.score(
    X_throughput_test.apply(le2.fit_transform), le2.fit_transform(y_throughput_test)
)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

mse = mean_squared_error(
    le2.fit_transform(y_throughput_test),
    hgbt_throughput.predict(X_throughput_test.apply(le2.fit_transform)),
)
print(f"The mean squared error (MSE) on test set: {mse:.4f}")

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

X = tmp.drop(
    columns=[
        "node_num",
        "on_node",
        "anomaly_hint_prob",
        "anomaly_hint_classification",
        "throughput_error_bar",
        "throughput_std",
        "throughput_relative_std",
        "time_error_bar",
        "time_std",
        "time_relative_std",
    ]
)
X_std: pd.DataFrame = StandardScaler().fit_transform(X.apply(le.fit_transform))

model = TSNE(n_components=3, random_state=0, max_iter=1000, n_jobs=-1)
model.set_output(transform="pandas")

tsne_df3: pd.DataFrame = model.fit_transform(X_std)

test = tmp
test.reset_index(drop=True, inplace=True)

tsne_df3 = test.join(tsne_df3)

model = TSNE(n_components=2, random_state=0, max_iter=1000, n_jobs=-1)
model.set_output(transform="pandas")

tsne_df2: pd.DataFrame = model.fit_transform(X_std)

test = tmp
test.reset_index(drop=True, inplace=True)

tsne_df2 = test.join(tsne_df2)

In [ ]:
import plotly.express as px

fig = px.scatter_3d(tsne_df3, x="tsne0", y="tsne1", z="tsne2", color="format")
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter(
    tsne_df2,
    x="tsne0",
    y="tsne1",
    color="format",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    tsne_df3,
    x="tsne0",
    y="tsne1",
    z="tsne2",
    color="node_num",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter(
    tsne_df2,
    x="tsne0",
    y="tsne1",
    color="node_num",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    tsne_df3,
    x="tsne0",
    y="tsne1",
    z="tsne2",
    color="parallel",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter(
    tsne_df2,
    x="tsne0",
    y="tsne1",
    color="parallel",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    tsne_df3,
    x="tsne0",
    y="tsne1",
    z="tsne2",
    color="total_filesize",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter(
    tsne_df2,
    x="tsne0",
    y="tsne1",
    color="total_filesize",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    tsne_df3[(tsne_df3["total_filesize"] == 10) ],
    x="tsne0",
    y="tsne1",
    z="tsne2",
    color="mean_throughput",
    symbol="format",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "total_filesize",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.update_traces(
    marker={"size": 12, "line": {"width": 2, "color": "DarkSlateGrey"}},
    selector={"mode": "markers"},
)
fig.show()

In [ ]:
import plotly.express as px

fig = px.scatter(
    tsne_df2[(tsne_df2["total_filesize"] == 10) ],
    x="tsne0",
    y="tsne1",
    color="mean_throughput",
    symbol="format",
    hover_data=[
        "mean_time",
        "mean_throughput",
        "total_filesize",
        "filesize_per_chunk",
        "num_ranks",
        "parallel",
        "format",
    ],
)
fig.update_traces(
    marker={"size": 12, "line": {"width": 2, "color": "DarkSlateGrey"}},
    selector={"mode": "markers"},
)
fig.show()